In [1]:
COUNTRIES     = ["USA", "GBR", "DEU", "FRA", "CHN", "JPN"]  
INDICATOR     = "NY.GDP.MKTP.CD"    
DATE_RANGE    = "2010:2024"         
PAGE_SIZE     = 1000                

StatementMeta(, 50e0300a-6164-4676-9b5a-a66df6ecc811, 3, Finished, Available, Finished, False)

In [2]:
import requests
import json
import os
from datetime import date, datetime

BRONZE_FILES = "/lakehouse/default/Files/gdp"
os.makedirs(BRONZE_FILES, exist_ok=True)

BASE_URL = "https://api.worldbank.org/v2/country/{iso}/indicator/{indicator}"

StatementMeta(, 50e0300a-6164-4676-9b5a-a66df6ecc811, 4, Finished, Available, Finished, False)

In [3]:
def fetch_wb_country(iso: str) -> list[dict]:
    url = BASE_URL.format(iso=iso, indicator=INDICATOR)
    params = {
        "format":   "json",
        "date":     DATE_RANGE,
        "per_page": PAGE_SIZE,
        "page":     1,
    }

    all_rows = []
    page     = 1

    while True:
        params["page"] = page
        print(f"    {iso} – page {page} …", end=" ")
        r = requests.get(url, params=params, timeout=30)
        r.raise_for_status()

        payload = r.json()
        if len(payload) < 2 or not payload[1]:
            print("no data")
            break

        meta  = payload[0]
        rows  = payload[1]
        total_pages = int(meta.get("pages", 1))

        print(f"{len(rows)} rows  (page {page}/{total_pages})")

        for row in rows:
            row["_country_iso"] = iso   # tag for traceability
        all_rows.extend(rows)

        if page >= total_pages:
            break
        page += 1

    return all_rows

StatementMeta(, 50e0300a-6164-4676-9b5a-a66df6ecc811, 5, Finished, Available, Finished, False)

In [4]:
print(f"\n{'='*60}")
print(f"  World Bank GDP Bronze Ingestion")
print(f"  Indicator : {INDICATOR}")
print(f"  Countries : {COUNTRIES}")
print(f"  Range     : {DATE_RANGE}")
print(f"  Started   : {datetime.utcnow().isoformat()}Z")
print(f"{'='*60}\n")

all_gdp_rows = []
for iso in COUNTRIES:
    print(f"\n  Fetching {iso} …")
    rows = fetch_wb_country(iso)
    all_gdp_rows.extend(rows)
    print(f"  → {len(rows)} observations for {iso}")

print(f"\n  Total rows collected: {len(all_gdp_rows):,}")

StatementMeta(, 50e0300a-6164-4676-9b5a-a66df6ecc811, 6, Finished, Available, Finished, False)


  World Bank GDP Bronze Ingestion
  Indicator : NY.GDP.MKTP.CD
  Countries : ['USA', 'GBR', 'DEU', 'FRA', 'CHN', 'JPN']
  Range     : 2010:2024
  Started   : 2026-05-22T06:29:23.360843Z


  Fetching USA …
    USA – page 1 … 15 rows  (page 1/1)
  → 15 observations for USA

  Fetching GBR …
    GBR – page 1 … 15 rows  (page 1/1)
  → 15 observations for GBR

  Fetching DEU …
    DEU – page 1 … 15 rows  (page 1/1)
  → 15 observations for DEU

  Fetching FRA …
    FRA – page 1 … 15 rows  (page 1/1)
  → 15 observations for FRA

  Fetching CHN …
    CHN – page 1 … 15 rows  (page 1/1)
  → 15 observations for CHN

  Fetching JPN …
    JPN – page 1 … 15 rows  (page 1/1)
  → 15 observations for JPN

  Total rows collected: 90


In [5]:
run_date  = date.today().isoformat()
out_file  = os.path.join(BRONZE_FILES, f"world_bank_gdp_{run_date}.json")

with open(out_file, "w", encoding="utf-8") as f:
    json.dump(all_gdp_rows, f, ensure_ascii=False, default=str)

size_kb = os.path.getsize(out_file) / 1024
print(f"\n  [OK] Written {out_file}  ({size_kb:.1f} KB)")

StatementMeta(, 50e0300a-6164-4676-9b5a-a66df6ecc811, 7, Finished, Available, Finished, False)


  [OK] Written /lakehouse/default/Files/gdp/world_bank_gdp_2026-05-22.json  (22.2 KB)


In [6]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import lit, current_timestamp

spark = SparkSession.builder.getOrCreate()

raw_df = (
    spark.read
    .option("multiLine", "true")
    .json(f"Files/gdp/world_bank_gdp_{run_date}.json")
    .withColumn("_ingested_at", current_timestamp())
    .withColumn("_source_file", lit(f"world_bank_gdp_{run_date}.json"))
)

print("\nSchema:")
raw_df.printSchema()
print(f"Rows: {raw_df.count():,}")

(
    raw_df.write
    .format("delta")
    .mode("append")
    .option("mergeSchema", "true")
    .saveAsTable("gdp_raw")
)

print("\n  [OK] Delta table 'gdp_raw' updated.")

StatementMeta(, 50e0300a-6164-4676-9b5a-a66df6ecc811, 8, Finished, Available, Finished, False)


Schema:
root
 |-- _country_iso: string (nullable = true)
 |-- country: struct (nullable = true)
 |    |-- id: string (nullable = true)
 |    |-- value: string (nullable = true)
 |-- countryiso3code: string (nullable = true)
 |-- date: string (nullable = true)
 |-- decimal: long (nullable = true)
 |-- indicator: struct (nullable = true)
 |    |-- id: string (nullable = true)
 |    |-- value: string (nullable = true)
 |-- obs_status: string (nullable = true)
 |-- unit: string (nullable = true)
 |-- value: double (nullable = true)
 |-- _ingested_at: timestamp (nullable = false)
 |-- _source_file: string (nullable = false)

Rows: 90

  [OK] Delta table 'gdp_raw' updated.


In [7]:
spark.sql("""
    SELECT _country_iso, countryiso3code, COUNT(*) AS years
    FROM gdp_raw
    GROUP BY 1, 2
    ORDER BY 1
""").show()

StatementMeta(, 50e0300a-6164-4676-9b5a-a66df6ecc811, 9, Finished, Available, Finished, False)

+------------+---------------+-----+
|_country_iso|countryiso3code|years|
+------------+---------------+-----+
|         CHN|            CHN|   15|
|         DEU|            DEU|   15|
|         FRA|            FRA|   15|
|         GBR|            GBR|   15|
|         JPN|            JPN|   15|
|         USA|            USA|   15|
+------------+---------------+-----+

